# SIMBES — Módulo 3: Gas y Flujo Multifásico

**Objetivo:** Comprender el impacto del gas libre en la operación de una bomba BES:  
1. Cálculo del GVF (Gas Volume Fraction) en la succión  
2. Umbral de gas lock (GVF > 15%)  
3. Degradación de la curva H-Q por gas  
4. Efecto de los separadores AGS  
5. Corrección de viscosidad (Hydraulic Institute)  

---
**Fuentes:** Standing (1977), SPE Brazil FATC 2022, HI ANSI/HI 9.6.7  
**Unidades:** sistema mixto campo (psi, ft, scf, STB, °F) — igual que frontend

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join('..', 'backend'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

from physics.gas import (
    gas_volume_fraction,
    hq_gas_degradation_factor,
    gas_separator_efficiency,
    viscosity_correction,
)

# Estilo coherente con tema oscuro de SIMBES
plt.rcParams.update({
    'figure.facecolor': '#0B0F1A',
    'axes.facecolor':   '#111827',
    'axes.edgecolor':   '#1E293B',
    'axes.labelcolor':  '#CBD5E1',
    'xtick.color':      '#64748B',
    'ytick.color':      '#64748B',
    'text.color':       '#CBD5E1',
    'grid.color':       '#1E293B',
    'grid.linestyle':   '--',
    'legend.facecolor': '#111827',
    'legend.edgecolor': '#1E293B',
    'font.family':      'monospace',
})
print('Imports OK ✓')

## 1. Gas Volume Fraction (GVF) en Succión

### Física

Cuando la presión de succión **Ps < Pb** (presión de burbuja), parte del gas disuelto se libera.  
La fracción volumétrica de gas libre en la succión define el régimen de operación:

$$
\text{GVF} = \frac{Q_{\text{gas}}}{Q_{\text{gas}} + Q_{\text{líquido}}}
$$

Donde, usando la aproximación de Standing (simplificada):

$$
R_s(P_s) \approx \text{GOR} \times \frac{P_s}{P_b} \quad [\text{scf/STB}]
$$

$$
\Delta R_s = \text{GOR} - R_s(P_s) \quad [\text{gas libre}]
$$

$$
B_g \approx \frac{0.02829 \cdot z \cdot T_R}{P_s} \quad [\text{ft}^3/\text{scf}]
$$

**Umbrales operativos:**
- GVF < 10% → operación segura  
- GVF 10–15% → zona de advertencia  
- GVF > 15% → **gas lock** — pérdida de succión

In [ ]:
# ─── Parámetros del pozo base (Ibis-12) ────────────────────────────
GOR   = 300    # scf/STB
Pb    = 1800   # psi — presión de burbuja
T_F   = 190    # °F  — temperatura de fondo

# Barrido de presión de succión
Ps_range = np.linspace(200, 2000, 300)
gvf_vals  = [gas_volume_fraction(GOR, Pb, Ps, T_F)['GVF_pct'] for Ps in Ps_range]

# Puntos de ejemplo
for Ps in [400, 800, 1200, 1800, 2200]:
    r = gas_volume_fraction(GOR, Pb, Ps, T_F)
    print(f"  Ps={Ps:5} psi → GVF = {r['GVF_pct']:5.1f}%  [{r['gas_lock_risk']}]")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(Ps_range, gvf_vals, color='#A78BFA', lw=2.5, label='GVF en succión')

# Zonas de riesgo
ax.axhspan(0,  10, alpha=0.12, color='#22C55E',  label='Seguro (< 10%)')
ax.axhspan(10, 15, alpha=0.12, color='#F59E0B',  label='Advertencia (10–15%)')
ax.axhspan(15, 50, alpha=0.12, color='#EF4444',  label='Gas Lock (> 15%)')

# Líneas de referencia
ax.axhline(15,  color='#EF4444', lw=1, ls='--', alpha=0.7)
ax.axhline(10,  color='#F59E0B', lw=1, ls='--', alpha=0.7)
ax.axvline(Pb,  color='#FBBF24', lw=1.5, ls=':', label=f'Pb = {Pb} psi')

ax.set_xlabel('Presión de Succión Ps [psi]')
ax.set_ylabel('GVF en Succión [%]')
ax.set_title(f'GVF vs Ps — Pozo Ibis-12   GOR={GOR} scf/STB · T={T_F}°F', pad=12)
ax.set_xlim(200, 2000)
ax.set_ylim(0, 50)
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## 2. Degradación de la Curva H-Q por Gas

El gas libre reduce la eficiencia hidráulica de cada etapa. El efecto se modela con un factor multiplicativo sobre la curva H-Q en agua:

$$
H_{\text{gas}}(Q) = f_H(\text{GVF}) \times H_{\text{limpia}}(Q)
$$

Factores empíricos representativos (datos de catálogo SLB/Baker Hughes):

| GVF | $f_H$ | Régimen |
|-----|-------|---------|
| 0–5% | 1.00 | Sin degradación |
| 5–10% | 0.90–1.00 | Leve |
| 10–15% | 0.70–0.90 | Moderada |
| >15% | <0.70 | Severa / Gas lock |

> **[SIMPLIFIED]** Factor uniforme aplicado a toda la curva. En la práctica la degradación varía por punto de operación.

In [ ]:
# Curva H-Q base (bomba representativa M3, 45 ft/etapa, 20 etapas, 60 Hz)
H0_STAGE   = 45    # ft por etapa
N_STAGES   = 20
H0_TOTAL   = H0_STAGE * N_STAGES   # ft cabeza máxima total
QMAX_STB   = 4200  # STB/d a 60 Hz
FT_TO_M    = 0.3048
STB_TO_M3  = 0.158987

Q_arr = np.linspace(0, QMAX_STB, 200)

def pump_head_ft(Q, freq=60):
    ratio = freq / 60
    Qref  = Q / ratio
    H_ref = H0_TOTAL * (1 - (Qref / QMAX_STB) ** 1.85)
    return max(0, H_ref * ratio**2)

H_clean = np.array([pump_head_ft(q) for q in Q_arr])

# GVF scenarios
gvf_scenarios = [
    (0.0,  '#38BDF8', 'GVF=0% (agua)'),
    (0.08, '#34D399', 'GVF=8%'),
    (0.12, '#F59E0B', 'GVF=12%'),
    (0.18, '#EF4444', 'GVF=18% (gas lock)'),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Panel izquierdo: curvas H-Q degradadas
for gvf, color, label in gvf_scenarios:
    deg = hq_gas_degradation_factor(gvf)
    fH  = deg['head_factor']
    ax1.plot(Q_arr * STB_TO_M3, H_clean * fH * FT_TO_M, color=color, lw=2.2, label=label)

ax1.set_xlabel('Caudal Q [m³/d]')
ax1.set_ylabel('Cabeza H [m]')
ax1.set_title('Degradación H-Q por GVF', pad=10)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.4)
ax1.set_xlim(0, QMAX_STB * STB_TO_M3)
ax1.set_ylim(0, H0_TOTAL * FT_TO_M * 1.05)

# Panel derecho: factores de degradación vs GVF
gvf_range = np.linspace(0, 0.30, 200)
fH_arr = [hq_gas_degradation_factor(g)['head_factor'] for g in gvf_range]
fQ_arr = [hq_gas_degradation_factor(g)['flow_factor'] for g in gvf_range]
fE_arr = [hq_gas_degradation_factor(g)['eff_factor']  for g in gvf_range]

ax2.plot(gvf_range*100, fH_arr, color='#38BDF8', lw=2, label='$f_H$ — Cabeza')
ax2.plot(gvf_range*100, fQ_arr, color='#34D399', lw=2, label='$f_Q$ — Caudal')
ax2.plot(gvf_range*100, fE_arr, color='#F472B6', lw=2, label='$f_{\\eta}$ — Eficiencia')
ax2.axvline(15, color='#EF4444', lw=1.5, ls='--', label='Umbral gas lock 15%')
ax2.set_xlabel('GVF en Succión [%]')
ax2.set_ylabel('Factor de corrección (0–1)')
ax2.set_title('Factores de Degradación vs GVF', pad=10)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.4)
ax2.set_xlim(0, 30)
ax2.set_ylim(0, 1.05)

plt.suptitle('Degradación de Curva H-Q por Gas Libre', color='#CBD5E1', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 3. Separadores de Gas — AGS y Gas Handler

Un separador instalado en la intake reduce el GVF que llega a la bomba:

$$
\text{GVF}_{\text{bomba}} = \text{GVF}_{\text{wellbore}} \times (1 - \eta_{\text{sep}})
$$

| Tipo | $\eta_{\text{sep}}$ | Aplicación típica |
|------|---------------------|-------------------|
| Sin separador | 0% | GVF < 8% |
| AGS Pasivo (Rotary) | 65% | GVF 8–20% |
| Gas Handler Activo | 82% | GVF > 20% |

In [ ]:
# Caso: GVF wellbore = 22%
GVF_WB = 0.22

print(f"GVF wellbore = {GVF_WB*100:.0f}%")
print("-" * 55)
for sep_type in ['none', 'ags_passive', 'gas_handler']:
    r = gas_separator_efficiency(GVF_WB, sep_type)
    print(f"  {r['separator_name']:<28} → GVF bomba = {r['GVF_pump_pct']:5.1f}%  "
          f"fH={r['head_factor']:.2f}  [{r['gas_lock_risk']}]")
    print(f"    {r['recommendation']}")

In [ ]:
# Mapa de GVF bomba vs GVF wellbore para los 3 tipos de separador
gvf_wb_range = np.linspace(0, 0.40, 200)
sep_types    = [
    ('none',        '#EF4444', 'Sin separador'),
    ('ags_passive',  '#F59E0B', 'AGS Pasivo (η=65%)'),
    ('gas_handler',  '#22C55E', 'Gas Handler (η=82%)'),
]

fig, ax = plt.subplots(figsize=(9, 5))

for sep_id, color, label in sep_types:
    gvf_pump = [gas_separator_efficiency(g, sep_id)['GVF_pump_pct'] for g in gvf_wb_range]
    ax.plot(gvf_wb_range * 100, gvf_pump, color=color, lw=2.2, label=label)

ax.axhline(15, color='#EF4444', lw=1.5, ls='--', alpha=0.7, label='Umbral gas lock 15%')
ax.axhline(10, color='#F59E0B', lw=1,   ls='--', alpha=0.7)
ax.axhspan(15, 45, alpha=0.08, color='#EF4444')
ax.axhspan(10, 15, alpha=0.08, color='#F59E0B')
ax.axhspan( 0, 10, alpha=0.08, color='#22C55E')

ax.set_xlabel('GVF Wellbore [%]')
ax.set_ylabel('GVF en Succión de la Bomba [%]')
ax.set_title('Reducción de GVF por Tipo de Separador', pad=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.4)
ax.set_xlim(0, 40)
ax.set_ylim(0, 40)
plt.tight_layout()
plt.show()

## 4. Corrección de Viscosidad — Hydraulic Institute (HI)

Para fluidos más viscosos que el agua, la curva H-Q se degrada. El método HI (ANSI/HI 9.6.7) define el parámetro de viscosidad:

$$
B = \frac{\sqrt{Q_{\text{BEP}}} \cdot H_{\text{BEP}}^{0.25}}{\sqrt{\nu}}
$$

Donde $\nu$ está en cSt (= cp para densidad ≈ 1). Los factores de corrección:

$$
Q_v = C_Q \times Q_w \quad ; \quad H_v = C_H \times H_w \quad ; \quad \eta_v = C_E \times \eta_w
$$

> Para 30 cp típico (crudos medianos): $C_Q \approx 0.86$, $C_H \approx 0.93$, $C_E \approx 0.74$

In [ ]:
# BEP de la bomba representativa
Q_BEP_GPM = 61.25   # gpm
H_BEP_FT  = 32.5    # ft/etapa en BEP

viscosidades = [1, 5, 10, 20, 30, 50, 100, 200]
print(f"{'μ [cp]':>8} {'B':>8} {'CQ':>6} {'CH':>6} {'CE':>6}")
print("-" * 45)
for mu in viscosidades:
    r = viscosity_correction(Q_BEP_GPM, H_BEP_FT, mu)
    B = r.get('viscosity_B', 999)
    print(f"{mu:>8}  {B:>8.1f}  {r['Cq']:>5.3f}  {r['Ch']:>5.3f}  {r['Ce']:>5.3f}")

In [ ]:
mu_range = np.linspace(1, 200, 300)
CQ_arr = [viscosity_correction(Q_BEP_GPM, H_BEP_FT, m)['Cq'] for m in mu_range]
CH_arr = [viscosity_correction(Q_BEP_GPM, H_BEP_FT, m)['Ch'] for m in mu_range]
CE_arr = [viscosity_correction(Q_BEP_GPM, H_BEP_FT, m)['Ce'] for m in mu_range]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(mu_range, CQ_arr, color='#38BDF8', lw=2, label='$C_Q$ — Caudal')
ax.plot(mu_range, CH_arr, color='#34D399', lw=2, label='$C_H$ — Cabeza')
ax.plot(mu_range, CE_arr, color='#F472B6', lw=2, label='$C_E$ — Eficiencia')

# Anotaciones para crudos típicos
for mu_mark, label in [(30, '30 cp\n(crudo medio)'), (100, '100 cp\n(crudo pesado)')]:
    r = viscosity_correction(Q_BEP_GPM, H_BEP_FT, mu_mark)
    ax.axvline(mu_mark, color='#64748B', lw=1, ls=':')
    ax.annotate(label, xy=(mu_mark, r['Ch']),
                xytext=(mu_mark + 5, r['Ch'] + 0.05),
                fontsize=8, color='#CBD5E1')

ax.set_xlabel('Viscosidad μ [cp]')
ax.set_ylabel('Factor de Corrección HI')
ax.set_title('Corrección de Viscosidad — Hydraulic Institute', pad=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.4)
ax.set_xlim(1, 200)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 5. Análisis Completo — Degradación Combinada Gas + Viscosidad

El factor total de degradación de la curva H-Q es:

$$
f_{H,\text{total}} = f_H(\text{GVF}_{\text{bomba}}) \times C_H(\mu)
$$

Este factor multiplica la curva H-Q en agua para obtener la curva efectiva en condiciones reales.

In [ ]:
# ─── Escenario Pozo Ibis-12 — comparación de 3 configuraciones ────
# Parámetros fijos
GOR   = 300    # scf/STB
Pb    = 1800   # psi
T_F   = 190    # °F
Ps    = 900    # psi — presión de succión nominal
mu    = 15     # cp

# GVF wellbore
gvf_res = gas_volume_fraction(GOR, Pb, Ps, T_F)
GVF_WB  = gvf_res['GVF']
print(f"GVF wellbore = {GVF_WB*100:.1f}%  ({gvf_res['gas_lock_risk']})")

configs = [
    ('none',        '#EF4444', 'Sin separador'),
    ('ags_passive',  '#F59E0B', 'AGS Pasivo (η=65%)'),
    ('gas_handler',  '#22C55E', 'Gas Handler (η=82%)'),
]

Q_arr  = np.linspace(0, QMAX_STB * STB_TO_M3, 200)
H_base = np.array([pump_head_ft(q / STB_TO_M3) * FT_TO_M for q in Q_arr])

# Curva de sistema (TDH aproximada)
depth_m  = 1800
Pwh_psi  = 150
rho_kgL  = 0.876
PSI_TO_M = 0.70307
TDH_sys  = depth_m + Pwh_psi * PSI_TO_M + 1.4e-5 * (Q_arr / STB_TO_M3)**2 * FT_TO_M * 0.5

fig, ax = plt.subplots(figsize=(10, 6))

# Curva limpia de referencia
ax.plot(Q_arr, H_base, color='#38BDF8', lw=2, ls='--', alpha=0.5, label='H-Q Limpia (referencia)')
# Curva de sistema
ax.plot(Q_arr, TDH_sys, color='#FB923C', lw=2.5, label='Curva Sistema (TDH)')

for sep_id, color, label in configs:
    sep_r    = gas_separator_efficiency(GVF_WB, sep_id)
    GVF_pump = sep_r['GVF_pump_pct'] / 100
    gas_deg  = hq_gas_degradation_factor(GVF_pump)
    visc_c   = viscosity_correction(Q_BEP_GPM, H_BEP_FT, mu)
    f_total  = gas_deg['head_factor'] * visc_c['Ch']

    H_deg = H_base * f_total
    ax.plot(Q_arr, H_deg, color=color, lw=2.2,
            label=f'{label} — GVF bomba={GVF_pump*100:.1f}% fH={f_total:.2f}')

ax.set_xlabel('Caudal Q [m³/d]')
ax.set_ylabel('Cabeza H [m]')
ax.set_title(f'Curvas H-Q Degradadas — Ibis-12\nGVF wellbore={GVF_WB*100:.1f}% · μ={mu} cp · Ps={Ps} psi',
             pad=10)
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.4)
ax.set_xlim(0, Q_arr[-1])
ax.set_ylim(0, max(H_base) * 1.05)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Mapa de calor: degradación total f_H en función de (GVF, viscosidad) ─
gvf_2d  = np.linspace(0, 0.30, 60)
mu_2d   = np.linspace(1,  150, 60)
GVF_G, MU_G = np.meshgrid(gvf_2d, mu_2d)

F_TOTAL = np.zeros_like(GVF_G)
for i in range(GVF_G.shape[0]):
    for j in range(GVF_G.shape[1]):
        gas_d = hq_gas_degradation_factor(GVF_G[i, j])
        visc_c = viscosity_correction(Q_BEP_GPM, H_BEP_FT, MU_G[i, j])
        F_TOTAL[i, j] = gas_d['head_factor'] * visc_c['Ch']

fig, ax = plt.subplots(figsize=(9, 6))
c = ax.contourf(GVF_G * 100, MU_G, F_TOTAL, levels=20, cmap='RdYlGn')
ax.contour(GVF_G * 100, MU_G, F_TOTAL, levels=[0.5, 0.7, 0.85], colors='white',
           linewidths=0.8, alpha=0.6)
ax.axvline(15, color='white', lw=2, ls='--', label='Gas lock GVF=15%')

cbar = plt.colorbar(c, ax=ax)
cbar.set_label('$f_{H,total}$ (factor degradación)', color='#CBD5E1')
cbar.ax.yaxis.set_tick_params(color='#64748B')

ax.set_xlabel('GVF en Succión [%]')
ax.set_ylabel('Viscosidad μ [cp]')
ax.set_title('Mapa de Degradación Total H-Q\n(Gas × Viscosidad)', pad=10)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 6. Resumen — Reglas Operativas M3

| Condición | Acción recomendada |
|-----------|--------------------|
| GVF wellbore < 8% | Operación normal sin separador |
| GVF wellbore 8–15% | Instalar AGS pasivo (η≈65%) |
| GVF wellbore 15–25% | AGS pasivo o Gas Handler activo (η≈82%) |
| GVF wellbore > 25% | Gas Handler activo obligatorio |
| μ > 30 cp | Aplicar corrección HI; sobredimensionar bomba |
| GVF > 15% **sin separador** | **PARAR** — riesgo inmediato de gas lock |

---
*Notebook generado automáticamente por SIMBES para el Módulo 3 — Gas y Flujo Multifásico.*  
*Versión: 1.0 · Fecha: 2026-03-12*